# `mln` tutorial: working with the PlanetNL population network

This notebook walks through the basics of the `MultiLayerNetwork` class from our `mln` package:
loading a network, looking at its parts, using the built-in functions, pulling out layers,
subsetting, and a few tailor-made recipes (mapping IDs, edge lists, affiliation matrices).

Run it in the CBS Research Access environment: the data (and the internal version of the `mln` package) are on the `H:` drive, the project-wide drive shared by everyone in the project.

# Getting started

## Starting Jupyter

From Git Bash, replacing `<project-id>` with your 4-digit CBS project number
(see [vscode_jupyter.md](vscode_jupyter.md) for more options):

```bash
/c/mambaforge/envs/<project-id>/Scripts/jupyter-notebook.exe --NotebookApp.notebook_dir="H:/"
```

## Imports

There are two versions of the package, and they are imported differently:

- **Internal version** (in `H:\shared_code`): `import sys`, add that folder to Python's search path, then `from mln import MultiLayerNetwork`.
- **Public version** (`mlnlib`): `from mlnlib.mln import MultiLayerNetwork`. No `import sys` or `sys.path` line needed.

The cell below uses the internal version. To use the public version, comment out the internal lines (including `import sys`) and uncomment the public one.

In [ ]:
import numpy as np                # arrays
import pandas as pd               # dataframes
import matplotlib.pyplot as plt   # plotting (used by .hist() below)
import scipy.sparse as sp         # sparse matrices (used for closure by hand below)

# --- internal version of the code ---
# make the mln package in H:\shared_code importable
# (sys is only needed for this, so only with the internal version)
import sys
sys.path.insert(0, "H:\\shared_code")
from mln import MultiLayerNetwork  # the main class: the population network

# --- public version of the package ---
# from mlnlib.mln import MultiLayerNetwork

# show floats with 3 decimals instead of scientific notation
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## Load the data

Networks are stored per year, and a year can have several versions of the network (subfolders such as `V2`).
Here we load version `V2` of the 2011 network from `H:\shared_data`. All three are arguments you can change:
set `data_dir`, `year` and `version` below to load from a different folder, year or version.

In [ ]:
data_dir = "H:\\shared_data"  # folder where the networks are stored
year = 2011                    # which year's network to load
version = "V2"                 # which version of the network to load, if the year has several

# folder with the saved network for that year and version
network_path = f"{data_dir}\\{year}\\{version}"

# load the saved network from disk
popnet = MultiLayerNetwork(
    load_from_library=True,     # read an existing network instead of building a new one
    library_path=network_path   # where to read it from
)  # polars=True <- try out with polars (then all pandas syntax below has to be rewritten in polars syntax)

## Elements of `popnet`

- `popnet.layers`: the link types (layers) in the network
- `popnet.A`: the full sparse adjacency matrix
- `popnet.nodes`: a dataframe with one row per person

In [ ]:
# table of all link types (layers) and the groups they belong to
popnet.layers

In [ ]:
# the full adjacency matrix: one row and one column per person, stored as a sparse matrix
popnet.A

In [ ]:
# number of people in the node table, in millions
print('All rows (millions): ', len(popnet.nodes) / 10**6)

# number of people flagged as active, in millions
print('Active (millions): ', len(popnet.nodes[popnet.nodes.active == True]) / 10**6)

# first 5 people, transposed so each column is a person and each row an attribute
popnet.nodes.head().T

## Direction of links: sources and targets

In principle, links go **from row to column**: the row id is the source, the column id is the target.
For the parents layer this means that a parent is the **row id** and their child is the **column id**.

**Beware:** CBS sometimes mixes up sources and targets. Be mindful of this when you work with a particular
link type, and check which way round its links actually go before relying on the direction.

**Direction also matters for sampled link types.** For layers such as colleagues and neighbours, direction tells you
who sampled whom: are these the colleagues or neighbours that ego has sampled (ego is the source, row),
or the ones that ego got sampled into (ego is the target, column)? Pick rows or columns accordingly.

## Example selections of people

The examples below use two selections of people:

- `selected_labels`: a few people by original ID (**label**), for the built-in functions that take `selected_nodes`.
  These are **hypothetical** placeholders. Real labels are privacy-sensitive: replace them with labels from
  `popnet.nodes` when you run the notebook, and never share or publish real ones.
- `random_list`: the first 10,000 people by matrix index (**id**), for cutting out smaller matrices.

In [ ]:
# hypothetical labels (original IDs) of two people: replace with real labels from popnet.nodes['label']
selected_labels = [123456789, 987654321]

# the first 10,000 people by matrix index (id)
random_list = list(range(0, 10000))

# Built-in functions

Functions can be restricted to selected people. `selected_nodes` takes original IDs (RIN), which the `MultiLayerNetwork` class calls **labels**.
Here we use the hypothetical `selected_labels` from *Example selections of people* above.
Without `selected_nodes`, the functions are computed for everyone.

## Degree

In [ ]:
# degree (number of ties) of every person, as a dictionary {label: degree}
popnet.get_degrees()

This is the ready-made function. A manual calculation of degrees (by summing the adjacency matrix)
is available further down, under *Tailor-made recipes: Degree by hand*.

Degree distributions are usually very skewed (many people with few ties, a few with very many),
so we plot them on a log-log scale. On a log scale the bins also need to be log-spaced,
otherwise almost all bins are squeezed into the right-hand side of the plot.

In [ ]:
# turn the {label: degree} dictionary into a pandas Series of degrees
degrees = pd.Series(popnet.get_degrees())

# log(0) is undefined, so leave out people with degree 0
degrees = degrees[degrees > 0]

# 100 bins evenly spaced on a log scale, from degree 1 to the maximum degree
bins = np.logspace(0, np.log10(degrees.max()), 100)

fig, ax = plt.subplots()
ax.hist(degrees, bins=bins)   # histogram of the degree distribution
ax.set_xscale('log')          # log scale on the x-axis
ax.set_yscale('log')          # log scale on the y-axis
ax.set_xlabel('Degree')
ax.set_ylabel('Number of people')
ax.set_title('Degree distribution (log-log)')
plt.show()

In [ ]:
# degrees of just these two people (original IDs, "labels" in the language of the class), as {label: degree}
popnet.get_degrees(
    selected_nodes=selected_labels
)

## Clustering coefficient

The clustering coefficient of a person is the share of pairs of their contacts who are also connected to each other.
A manual implementation with matrix multiplication is shown further below (*Tailor-made recipes: Closure by hand*).

In [ ]:
# clustering coefficient of these two people:
# the share of their contacts who are also connected to each other
popnet.get_clustering_coefficient(
    selected_nodes=selected_labels
)

## Excess closure

**Slow:** `get_excess_closure` runs preliminary calculations for the whole network,
even when you pass only a few selected people, so it takes a long time.

In [ ]:
# excess closure of these two people (slow, see above)
popnet.get_excess_closure(
    selected_nodes=selected_labels
)

# Working with the matrices

Getting matrices out of `popnet`: single layers, all ties, and smaller pieces of the matrix.

## Get all ties as a binary matrix

`get_binary_adjacency` collapses all layers into one 0/1 matrix.

In [ ]:
# all ties of all layers in one matrix: 1 if two people have any tie, 0 otherwise
A = popnet.get_binary_adjacency()

## Get one link type or one layer

`get_layer_adjacency_matrix` returns the adjacency matrix for a single layer (by layer code, e.g. `301` for parents)
or for a group of layers (by group name, e.g. `'family'`).

**Beware:** `work` is misspelled in the data, so check `popnet.layers` for the exact name.

In [ ]:
# show the documentation of the function, including what each argument does
help(popnet.get_layer_adjacency_matrix)

In [ ]:
# adjacency matrix of only the parent ties
parents = popnet.get_layer_adjacency_matrix(
    301,      # layer code 301 = parents
    'layer',  # 301 is a single layer code (not a group name)
    False,    # see help() above
    'bool'    # return True/False values
)

# adjacency matrix of all family ties together
F = popnet.get_layer_adjacency_matrix(
    'family', # group name: all layers in the family group
    'group',  # 'family' is a group name (not a layer code)
    False,    # see help() above
    'bool'    # return True/False values
)

In [ ]:
# first 10 rows and columns, converted from sparse to a normal (dense) matrix to look at
parents[:10, :10].todense()

## Cut out a square matrix for a subset of people

`get_filtered_network` returns a new network with only the people in `nodes_selected`, as both rows *and* columns.
So it contains only the connections **between these selected people**, and no one else: ties from or to
people outside the selection are dropped. (To keep those ties, take a long slice instead, see below.)

In [ ]:
# new network with only these people: a 10,000 x 10,000 matrix
# containing only the ties among them (ties to anyone else are dropped)
popnet_square = popnet.get_filtered_network(
    nodes_selected=random_list,  # people to keep
    node_type='id'               # the list contains matrix indices (id), not original IDs (label)
)

# the adjacency matrix of the smaller network
popnet_square.A

## Get a long slice of the matrix

Selecting only some rows (or only some columns) keeps the other dimension whole: the selected people and their ties to *everyone*.

**Be mindful of directionality:** do you want the ties that go **from** these people, or the ties that go **to** these people?

- **rows** (`A[random_list, :]`): ties going *from* the selected people (they are the source)
- **columns** (`A[:, random_list]`): ties going *to* the selected people (they are the target)

For a symmetric link type both give the same ties; for a directed one (such as parents, or sampled colleagues and neighbours) they do not.

In [ ]:
# rows for the selected people, all columns: 10,000 x (all people)
# = ties going FROM the selected people
popnet_long = popnet.A[random_list, :]

popnet_long

In [ ]:
# columns for the selected people, all rows: (all people) x 10,000
# = ties going TO the selected people
popnet_long_to = popnet.A[:, random_list]

popnet_long_to

# Tailor-made recipes

## Linking the network to other data

Turning the network into an edge list, and using the ID dictionaries to bring in characteristics of people.

### Convert the network to a dataframe (edge list)

This uses `A` from *Get all ties as a binary matrix* above, so it includes all ties. Always check directionality!
`person` is the source (row id) and `connected_person` the target (column id), keeping in mind that CBS sometimes mixes these up.

An edge list is handy for mapping people's characteristics onto both ends of each tie (see below).

In [ ]:
# A.nonzero() gives (row indices, column indices) of all ties;
# stack them as two columns: one row per tie
popnet_df = pd.DataFrame(
    np.array(A.nonzero()).T,
    columns=['person', 'connected_person']  # row id = source, column id = target
)

popnet_df.head()

### Dictionaries to map people to other datasets

`label` is the original ID (RIN), `id` is the row/column index in the matrix.

These dictionaries are important: they are how you link the network to everything else about people.
Other microdata sources and external data are keyed by the original ID (`label`), while the network matrices
use the matrix index (`id`). Use `nodemap_dict` to bring attributes from other microdata files or external data
into the network, and `nodemap_dict_back` to take results from the network back to those sources.

In [ ]:
# original ID (label) -> matrix index (id)
nodemap_dict = (
    popnet.nodes[['label', 'id']]  # keep only the two ID columns
    .set_index('label')             # use label as the key
    .to_dict()['id']                # turn into {label: id}
)

# matrix index (id) -> original ID (label)
nodemap_dict_back = (
    popnet.nodes[['label', 'id']]
    .set_index('id')                # use id as the key
    .to_dict()['label']             # turn into {id: label}
)

### Map people's characteristics onto the edge list

The edge list contains matrix ids, so we build a dictionary `{id: characteristic}`, the same way as
`nodemap_dict_back` above, and map it onto both columns: the ego (`person`) and the alter (`connected_person`).
Here the characteristic is birth year.

In [ ]:
# name of the birth year column in popnet.nodes: check popnet.nodes.columns for the exact name
birth_year_col = 'birth_year'

# matrix index (id) -> birth year
birthyear_dict = (
    popnet.nodes[['id', birth_year_col]]
    .set_index('id')                # use id as the key
    .to_dict()[birth_year_col]      # turn into {id: birth year}
)

# birth year of the ego (source of the tie)
popnet_df['person_birth_year'] = popnet_df['person'].map(birthyear_dict)

# birth year of the alter (target of the tie)
popnet_df['connected_person_birth_year'] = popnet_df['connected_person'].map(birthyear_dict)

popnet_df.head()

## Operations on matrices

Computing degrees yourself, and removing or adding layers.

### Degree by hand

`get_degrees()` (see *Built-in functions* above) is the ready-made way to get degrees. You can also compute degrees yourself
by summing the adjacency matrix, which is useful when you want degrees for your own version of the matrix
(for example one layer only, or all ties except parents, as below).

Keep in mind that the network is **directed**, so summing over columns and summing over rows give different results.
Links go from row (source) to column (target), but CBS sometimes mixes this up for particular link types
(see *Direction of links* above).

- **column sums** (`axis=0`): the number of links pointing *to* each person (in-degree)
- **row sums** (`axis=1`): the number of links going *from* each person (out-degree)

Unlike `get_degrees()`, the results are ordered by matrix index (`id`), not keyed by `label`.

In [ ]:
# all ties of all layers in one 0/1 matrix (see *Get all ties as a binary matrix* above)
A_all = popnet.get_binary_adjacency()

# column sums: for each person, the number of links that point to them (in-degree)
in_degree = A_all.sum(axis=0)

# row sums: for each person, the number of links that go out from them (out-degree)
out_degree = A_all.sum(axis=1)

in_degree

### Subtracting matrices: removing a layer

Matrices of the same people can be subtracted cell by cell. Subtracting the parents matrix (from *Get one link type or one layer*) from the matrix of
all ties `A` (from *Get all ties as a binary matrix*) removes the parent ties, leaving all other ties.

After the subtraction, a cell can be `-1` where there is a parent tie but no tie in `A`, so we set negative values
to `0`. The opposite operation, adding layers together, is shown next.

In [ ]:
# remove parent ties: subtract the parents matrix
A = A - parents

# where there was a parent tie but no tie in A, the subtraction gives -1: set those to 0
A.data[A.data < 0] = 0

# drop the stored zeros so the sparse matrix stays small
A.eliminate_zeros()

### Adding layers together: school and neighbours

Two adjacency matrices of the same people can be summed cell by cell. Here we take the school and neighbours layers,
subsample them to the 10,000 people selected above (to keep the example small), and add them together.

After the sum, each cell tells you in how many of the two layers a tie exists:

- `0`: no tie
- `1`: a tie in one of the two layers (school or neighbours)
- `2`: a tie in both layers

**Beware:** convert the `'bool'` matrices to integers before summing. For True/False matrices `+` means "or",
so the sum would only ever be True/False and you could not see where a tie exists in both layers.

In [ ]:
# group names of the two layers: check popnet.layers for the exact names
school_group = 'school'
neighbour_group = 'neighbours'

# binary (True/False) matrices for the two layers
school = popnet.get_layer_adjacency_matrix(school_group, 'group', False, 'bool')
neighbours = popnet.get_layer_adjacency_matrix(neighbour_group, 'group', False, 'bool')

# subsample: ties among the 10,000 selected people only,
# converted from True/False to 0/1 integers so that summing counts
school_sub = school[random_list, :][:, random_list].astype(np.int64)
neighbours_sub = neighbours[random_list, :][:, random_list].astype(np.int64)

# sum of the two layers: 0 = no tie, 1 = tie in one layer, 2 = tie in both
combined = school_sub + neighbours_sub

# how many cells have each value (only non-zero cells are stored in a sparse matrix)
pd.Series(combined.data).value_counts().sort_index()

## Matrix multiplication

### `*` versus `@`

Python has two multiplication operators for matrices, and they do different things:

- `@` is always the **matrix product**: row $i$ of the first matrix times column $j$ of the second, summed.
  This is what the formulas in network analysis mean (paths of length 2, projecting people onto groups, ...).
- `*` depends on the type of the object:
  - for numpy arrays and scipy sparse *arrays* (e.g. `csr_array`) it is **element-wise**: cell $(i, j)$ times cell $(i, j)$
  - for `np.matrix` and scipy sparse *matrices* (e.g. `csr_matrix`) it is the **matrix product**

So the same `X * Y` can silently give a different result depending on what type `X` and `Y` are.
For our purposes we always use `@`, because it means matrix product whatever the type.
When we do want element-wise multiplication of sparse matrices, we write it explicitly with `.multiply()`.

In [ ]:
# a small example with numpy arrays
X = np.array([[1, 2],
              [3, 4]])
Y = np.array([[0, 1],
              [1, 0]])

# element-wise: each cell times the same cell -> [[0, 2], [3, 0]]
print(X * Y)

# matrix product: rows times columns -> [[2, 1], [4, 3]]
print(X @ Y)

### Closure by hand

Closure (clustering) asks: of all pairs of a person's contacts, how many are connected to each other?
With a symmetric 0/1 matrix `S` this can be computed with matrix multiplication:

- $(S \cdot S)_{ij}$ is the number of contacts that $i$ and $j$ have in common (paths of length 2)
- keeping only the pairs that are themselves tied (element-wise product with $S$) leaves the closed paths, i.e. triangles
- summing row $i$ gives twice the number of triangles person $i$ is part of

For a person with degree $k_i$ and $T_i$ triangles:

$$C_i = \frac{T_i}{k_i (k_i - 1) / 2} \qquad \text{overall closure} = \frac{\sum_i T_i}{\sum_i k_i (k_i - 1) / 2}$$

**Notes:**
- Closure is defined on undirected ties, so we first make the matrix symmetric: a tie counts if it goes in either direction.
- `S @ S` on the whole population network needs a lot of memory, so we demonstrate it on the 10,000-person subset.
  Ties to people outside the subset are dropped, so the values differ from those on the full network.
- `get_clustering_coefficient` may handle direction or layers differently, so its values need not match exactly.

In [ ]:
# 0/1 matrix of all ties among the 10,000 selected people (rows and columns)
S = A_all[random_list, :][:, random_list]

# make it undirected: a tie counts if it goes in either direction, stored as 0/1 integers
S = ((S + S.T) > 0).astype(np.int64)

# remove self-ties on the diagonal, if any
S = S - sp.diags(S.diagonal(), dtype=np.int64)
S.eliminate_zeros()

# degree of each person in the undirected subset
k = np.asarray(S.sum(axis=1)).ravel()

# (S @ S)[i, j] = number of common contacts of i and j (matrix product: @);
# .multiply(S) keeps only pairs i, j that are tied themselves = closed triangles (element-wise)
closed = (S @ S).multiply(S)

# each triangle of person i is counted twice in row i (once via each of the other two people)
triangles = np.asarray(closed.sum(axis=1)).ravel() / 2

# number of possible pairs among each person's contacts
pairs = k * (k - 1) / 2

# local clustering coefficient per person (NaN for people with fewer than 2 contacts)
local_clustering = np.divide(
    triangles, pairs,
    out=np.full(len(pairs), np.nan),
    where=pairs > 0
)

# overall closure: all closed pairs divided by all possible pairs
overall_closure = triangles.sum() / pairs.sum()

print('Overall closure: ', overall_closure)
pd.Series(local_clustering).describe()

### Multiplexity: ties that exist in more than one layer

A tie is **multiplex** when the same pair of people is connected in more than one layer,
for example they went to the same school *and* are neighbours.

To find these ties, multiply the two 0/1 matrices **element-wise** with `.multiply()` (not `@`, see
*`*` versus `@`* above): a cell is 1 only where both matrices have a 1.
This gives the same cells as `combined == 2` above.

**Keep directionality in mind:** a school tie $i \to j$ and a neighbour tie $j \to i$ are in different cells,
so they do not count as the same edge. If direction does not matter for your question, make both matrices
undirected first (as in *closure by hand* above).

In [ ]:
# 1 where the tie exists in both layers, 0 elsewhere (element-wise product)
both = school_sub.multiply(neighbours_sub).tocsr()
both.eliminate_zeros()

# number of ties that are both school and neighbour ties
n_both = both.nnz
print('Ties in both layers: ', n_both)

# share of school ties that are also neighbour ties, and the other way round
print('Share of school ties that are also neighbour ties: ', n_both / school_sub.nnz)
print('Share of neighbour ties that are also school ties: ', n_both / neighbours_sub.nnz)

# the multiplex ties as an edge list (matrix indices within the 10,000-person subset)
multiplex_df = pd.DataFrame(
    np.array(both.nonzero()).T,
    columns=['person', 'connected_person']  # row id = source, column id = target
)

multiplex_df.head()

## Affiliation matrices

Linking people to groups, and projecting the person network onto those groups.

### Creating an affiliation matrix

An affiliation matrix links people to groups. A group can be defined by anything: neighbourhood, age cohort,
migrant generation, socioeconomic status, and so on. All you need is a correspondence between people and groups,
either readily available (for example a column in `popnet.nodes`) or created from other microdata files.

Here the groups are neighbourhoods (`buurt_code`). We only use active people with a known neighbourhood.

In [ ]:
# keep active people with a known neighbourhood
filtered = popnet.nodes[
    (popnet.nodes.active == True) &
    (popnet.nodes["buurt_code"].notna())
].copy()

# list of (original ID, neighbourhood code) pairs
buurt_dict = list(
    zip(
        filtered['label'],
        filtered['buurt_code']
    )
)

# build a person x neighbourhood matrix and store it under the name "buurt_affiliation"
popnet.create_affiliation_matrix(
    "buurt_affiliation",  # name to store it under
    buurt_dict            # the (person, neighbourhood) pairs
)

`create_affiliation_matrix` stores more than the matrix itself. Next to the matrix (`"A"`), it creates a new
**dictionary that maps the new rows/columns to the groups**, here to neighbourhoods (buurt codes).
The people keep their usual matrix index (`id`), but the neighbourhoods get new indices of their own.
You need this dictionary to know which neighbourhood a given row or column of the result belongs to,
for example in `B_B` below.

The dictionary is stored under its own name inside `popnet.affiliation_matrix["buurt_affiliation"]`.
To find that name, list what is stored with `.keys()`, then access the dictionary with
`popnet.affiliation_matrix["buurt_affiliation"]["<its name>"]`.

In [ ]:
# everything stored for this affiliation
popnet.affiliation_matrix["buurt_affiliation"]

In [ ]:
# list the names of everything stored for this affiliation;
# one of them is "A" (the matrix), another is the dictionary that maps the new indices to neighbourhoods
popnet.affiliation_matrix["buurt_affiliation"].keys()

In [ ]:
# the dictionary that maps the new indices to neighbourhoods (buurt codes):
# replace "..." with its name from the keys() output above, then uncomment
# buurt_index_dict = popnet.affiliation_matrix["buurt_affiliation"]["..."]

In [ ]:
# the affiliation matrix itself: people (rows) x neighbourhoods (columns)
popnet.affiliation_matrix["buurt_affiliation"]["A"]

### From a person network to a neighbourhood network

With `P_P` the person-to-person matrix and `A` the person-to-neighbourhood affiliation matrix:

$$B\_B = A^T \cdot P\_P \cdot A$$

Dimensions: $(B \times P) \cdot (P \times P) \cdot (P \times B) = (B \times B)$

**Reading the dimensions.** Each bracket is the shape of one matrix, written as (number of rows $\times$ number of columns),
with $P$ = the number of people and $B$ = the number of neighbourhoods (buurten):

- $A^T$ is $(B \times P)$: one row per neighbourhood, one column per person (the affiliation matrix transposed)
- $P\_P$ is $(P \times P)$: one row and one column per person
- $A$ is $(P \times B)$: one row per person, one column per neighbourhood

A matrix product `X @ Y` only works when the number of columns of `X` equals the number of rows of `Y`
(the inner dimensions match). The result has the rows of `X` and the columns of `Y` (the outer dimensions).
So the $P$'s in the middle "cancel out" and what is left is $(B \times B)$: a neighbourhood-by-neighbourhood matrix.
Writing out the dimensions like this is a quick way to check that a chain of matrix products makes sense
before you run it.

Each cell of `B_B` counts the ties between people in neighbourhood *i* and people in neighbourhood *j*.

In [ ]:
# person x person: all ties, binary
P_P = popnet.get_binary_adjacency()

# dimensions as (rows x columns), P = people, B = neighbourhoods:
# (P x P) @ (P x B) = (P x B): for each person, the number of ties into each neighbourhood
interim_M = (
    P_P
    @ popnet.affiliation_matrix["buurt_affiliation"]["A"]
)

# (B x P) @ (P x B) = (B x B): ties between neighbourhoods
B_B = (
    popnet.affiliation_matrix["buurt_affiliation"]["A"].T
    @ interim_M
)

B_B